# Overfitting 测试

选择 100 个 image_id，用其对应的切片进行过拟合训练，验证数据 pipeline 正确性。

**使用配置：** `configs/train_debug.yaml`（与正式训练使用相同的增强配置）

**判定标准：**
- 训练 Loss 迅速下降，接近 0
- 训练 mAP 接近 1.0 (100%)

**如果失败：** 说明数据 pipeline 有 Bug（标签格式、坐标偏移、归一化等）


In [5]:
import os
import random
import shutil
from pathlib import Path

import yaml


In [6]:
# 路径配置
SLICED_ROOT = Path("/Users/weixianfu/Documents/Datas/mtsd-resized")
TRAIN_IMAGES_DIR = SLICED_ROOT / "train_full" / "images"
TRAIN_LABELS_DIR = SLICED_ROOT / "train_full" / "labels"

# Overfitting 子集输出目录
OVERFIT_DIR = Path("/Users/weixianfu/Documents/Datas/mtsd-overfit")
OVERFIT_IMAGES = OVERFIT_DIR / "images"
OVERFIT_LABELS = OVERFIT_DIR / "labels"

# 参数
NUM_IMAGES = 100  # 选择的原图数量
RANDOM_SEED = 42


In [7]:
# Step 1: 收集所有原图 ID（从切片文件名中提取）
all_files = list(TRAIN_IMAGES_DIR.glob("*.jpg"))

# 提取原图 ID（去掉 _0, _1, _F 后缀）
image_ids = set()
for f in all_files:
    name = f.stem
    # 去掉最后的 _数字 或 _F 后缀
    if name.endswith("_F"):
        base_id = name[:-2]
    elif "_" in name:
        parts = name.rsplit("_", 1)
        if parts[-1].isdigit() or parts[-1] == "F":
            base_id = parts[0]
        else:
            base_id = name
    else:
        base_id = name
    image_ids.add(base_id)

image_ids = list(image_ids)
print(f"总共找到 {len(image_ids)} 个原图 ID")

# 随机选择 100 个
random.seed(RANDOM_SEED)
selected_ids = random.sample(image_ids, min(NUM_IMAGES, len(image_ids)))
print(f"随机选择了 {len(selected_ids)} 个原图 ID")


总共找到 32782 个原图 ID
随机选择了 100 个原图 ID


In [8]:
# Step 2: 复制选中 ID 的所有切片到 overfit 目录
# 清空并重建目录
if OVERFIT_DIR.exists():
    shutil.rmtree(OVERFIT_DIR)
OVERFIT_IMAGES.mkdir(parents=True)
OVERFIT_LABELS.mkdir(parents=True)

copied_images = 0
copied_labels = 0

for image_id in selected_ids:
    # 查找该 ID 的所有切片（_0, _1, _2, ... 和 _F）
    for img_file in TRAIN_IMAGES_DIR.glob(f"{image_id}_*.jpg"):
        # 复制图片
        shutil.copy(img_file, OVERFIT_IMAGES / img_file.name)
        copied_images += 1
        
        # 复制对应的 label（如果存在）
        label_file = TRAIN_LABELS_DIR / f"{img_file.stem}.txt"
        if label_file.exists():
            shutil.copy(label_file, OVERFIT_LABELS / label_file.name)
            copied_labels += 1

print(f"复制完成:")
print(f"  - 图片: {copied_images}")
print(f"  - 标签: {copied_labels}")


复制完成:
  - 图片: 413
  - 标签: 413


In [ ]:
# Step 3: 检查 Overfit 标签中的类别 ID 范围
min_id = 1e9
max_id = -1
bad_files = []

for lbl in OVERFIT_LABELS.glob("*.txt"):
    with open(lbl, "r") as f:
        for line in f:
            parts = line.strip().split()
            if not parts:
                continue
            cid = int(parts[0])
            if cid < min_id:
                min_id = cid
            if cid > max_id:
                max_id = cid
            if cid < 0 or cid >= 401:
                bad_files.append((lbl, cid))

print(f"min class id: {min_id}")
print(f"max class id: {max_id}")
print(f"bad label count: {len(bad_files)}")
print("examples:")
for p, cid in bad_files[:10]:
    print(f"  {p.name}: {cid}")


In [9]:
# Step 3: 读取原始 data.yaml 获取类别信息，生成 overfit 用的 data.yaml
PROJECT_ROOT = Path.cwd().parent
original_data_yaml = PROJECT_ROOT / "configs" / "data.yaml"

with open(original_data_yaml, "r") as f:
    original_cfg = yaml.safe_load(f)

# 创建 overfit 用的 data.yaml
# 注意：train 和 val 都指向同一目录，用于过拟合测试
overfit_data_cfg = {
    "path": str(OVERFIT_DIR),
    "train": "images",
    "val": "images",  # 验证集也用训练集，用于检测过拟合
    "nc": original_cfg.get("nc", 401),  # 类别数
    "names": original_cfg.get("names", {}),
}

overfit_data_yaml = OVERFIT_DIR / "data.yaml"
with open(overfit_data_yaml, "w") as f:
    yaml.dump(overfit_data_cfg, f, default_flow_style=False)

print(f"生成 data.yaml: {overfit_data_yaml}")
print(f"  - path: {overfit_data_cfg['path']}")
print(f"  - train: {overfit_data_cfg['train']}")
print(f"  - val: {overfit_data_cfg['val']}")
print(f"  - nc: {overfit_data_cfg['nc']}")


生成 data.yaml: /Users/weixianfu/Documents/Datas/mtsd-overfit/data.yaml
  - path: /Users/weixianfu/Documents/Datas/mtsd-overfit
  - train: images
  - val: images
  - nc: 401


In [10]:
# Step 4: 加载 train_debug.yaml 配置并开始训练
from ultralytics import YOLO

# 加载训练配置
train_config_path = PROJECT_ROOT / "configs" / "train_debug.yaml"
with open(train_config_path, "r") as f:
    train_cfg = yaml.safe_load(f)

# 提取模型名称
model_name = train_cfg.pop("model", "yolov8n.pt")
model = YOLO(model_name)

# 覆盖 overfit 专用设置
train_cfg["data"] = str(overfit_data_yaml)
train_cfg["project"] = "runs/overfit"
train_cfg["name"] = f"{Path(model_name).stem}_overfit_{NUM_IMAGES}"

print(f"使用配置: {train_config_path}")
print(f"模型: {model_name}")
print(f"输出: {train_cfg['project']}/{train_cfg['name']}")

# 开始训练
results = model.train(**train_cfg)


使用配置: /Users/weixianfu/Documents/Projects/road-sign-eu-mvp/configs/train_debug.yaml
模型: yolov8s.pt
输出: runs/overfit/yolov8s_overfit_100
New https://pypi.org/project/ultralytics/8.3.240 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.234 🚀 Python-3.10.19 torch-2.9.1 MPS (Apple M3 Max)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=12, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/weixianfu/Documents/Datas/mtsd-overfit/data.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_rati

RuntimeError: shape mismatch: value tensor of shape [2088] cannot be broadcast to indexing result of shape [2740]

In [ ]:
# Step 5: 检查过拟合结果
print("=" * 60)
print("过拟合测试结果")
print("=" * 60)

# 获取最终指标
final_metrics = results.results_dict

print(f"\n最终训练指标:")
print(f"  - mAP50:    {final_metrics.get('metrics/mAP50(B)', 'N/A')}")
print(f"  - mAP50-95: {final_metrics.get('metrics/mAP50-95(B)', 'N/A')}")
print(f"  - Precision: {final_metrics.get('metrics/precision(B)', 'N/A')}")
print(f"  - Recall:    {final_metrics.get('metrics/recall(B)', 'N/A')}")

print("\n" + "=" * 60)
print("判定标准:")
print("  ✅ mAP50 > 0.9 → 数据 pipeline 正确")
print("  ❌ mAP50 < 0.9 → 数据 pipeline 有 Bug，需要排查")
print("=" * 60)

mAP50 = final_metrics.get('metrics/mAP50(B)', 0)
if mAP50 > 0.9:
    print(f"\n🎉 过拟合测试通过！mAP50 = {mAP50:.4f}")
    print("数据加载、标签格式、切片逻辑验证正确。")
    print("可以进行正式训练。")
else:
    print(f"\n⚠️ 过拟合测试未通过！mAP50 = {mAP50:.4f}")
    print("请检查:")
    print("  1. 标签格式是否为 YOLO 格式 (cls cx cy w h)")
    print("  2. 坐标是否归一化到 [0, 1]")
    print("  3. 切片逻辑是否正确映射了 bbox")


In [ ]:
# Step 6 (可选): 可视化预测结果
import cv2
import matplotlib.pyplot as plt

# 加载训练好的模型（使用上面训练的输出路径）
best_weights = Path(train_cfg["project"]) / train_cfg["name"] / "weights" / "best.pt"
best_model = YOLO(str(best_weights))
print(f"加载模型: {best_weights}")

# 随机选择一张图进行预测
test_images = list(OVERFIT_IMAGES.glob("*.jpg"))
test_img = random.choice(test_images)

# 预测
results = best_model.predict(str(test_img), conf=0.5, verbose=False)
result = results[0]

# 绘制
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 原图
img = cv2.imread(str(test_img))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
axes[0].imshow(img_rgb)
axes[0].set_title(f"Original: {test_img.name}")
axes[0].axis("off")

# 预测结果
annotated = result.plot()
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
axes[1].imshow(annotated_rgb)
axes[1].set_title(f"Predictions: {len(result.boxes)} boxes")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print(f"\n检测到 {len(result.boxes)} 个目标")
